In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_parquet(
    "../../trained_models/test4/predictions_best_val_loss/ALL_predictions.parquet"
)
df["probabilities"] = df["logits"].apply(lambda x: np.exp(x) / np.sum(np.exp(x)))
df["predicted_label_probability"] = df["probabilities"].apply(lambda x: np.max(x))
df_test = df[df["stage"] == "test"]
LABEL_ENCODING = {
    "healthy": 0,
    "scab": 1,
    "rust": 2,
    "frog_eye_leaf_spot": 3,
    "powdery_mildew": 4,
}

In [3]:
df_test.head()

,id,image_path,label,stage,true_label,predicted_label,logits,embedding,probabilities,predicted_label_probability
12540,plant_pathology_12540,f4e0f3d6cdc21449.jpg,healthy,test,0,0,"[5.9578905, 0.42065713, -5.507152, -1.443414, ...","[0.8678111, 0.11397629, 0.80153644, 1.0588629,...","[0.9952267, 0.0039186105, 1.04404035e-05, 0.00...",0.995227
12541,plant_pathology_12541,ef6d90e1c01ded90.jpg,powdery_mildew,test,4,4,"[-1.5356348, -1.4539038, -3.7593925, -1.628738...","[1.6914755, 0.4146839, 0.0, 2.3988786, 0.19858...","[0.00025650728, 0.0002783524, 2.7754535e-05, 0...",0.999204
12542,plant_pathology_12542,9aac534b4bb624f4.jpg,frog_eye_leaf_spot,test,3,3,"[-11.074297, 2.3991048, 2.7187505, 13.330098, ...","[0.2655799, 1.0805931, 3.70922, 0.09137227, 0....","[2.5193436e-11, 1.7894166e-05, 2.463382e-05, 0...",0.999957
12543,plant_pathology_12543,891b99b0617e6567.jpg,healthy,test,0,0,"[6.1893497, -0.29004213, -5.5005617, -1.336389...","[0.9056382, 0.12818289, 0.7405, 1.2189583, 1.0...","[0.99769557, 0.0015312072, 8.358609e-06, 0.000...",0.997696
12544,plant_pathology_12544,fa8f7558aa489694.jpg,rust,test,2,2,"[-8.103739, -2.636804, 12.877483, 5.4338217, -...","[0.03275104, 2.398438, 0.900273, 0.023848446, ...","[7.72177e-10, 1.8280012e-07, 0.99941504, 0.000...",0.999415


In [6]:
df_test["predicted_label"] = np.where(df_test["predicted_label_probability"] > 0.5, df_test["predicted_label"], -1)

/var/folders/s0/3d36mhrj5fddxzr1l8vz89q40000gn/T/ipykernel_97705/4055372408.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["predicted_label"] = np.where(df_test["predicted_label_probability"] > 0.5, df_test["predicted_label"], -1)


In [7]:
for label in LABEL_ENCODING.keys():
    df_label = df_test[df_test["true_label"] == LABEL_ENCODING[label]]
    correct = df_label[df_label["predicted_label"] == df_label["true_label"]]
    total = len(df_label)
    labeled_ood = len(df_label[df_label["predicted_label"] == -1])
    print(f"Label: {label}")
    print(f"  Total samples: {total}")
    print(f"  Correctly classified: {len(correct)} ({len(correct)/total:.2%})")
    print(f"  Labeled as OOD: {labeled_ood} ({labeled_ood/total:.2%})")

Label: healthy
  Total samples: 925
  Correctly classified: 827 (89.41%)
  Labeled as OOD: 24 (2.59%)
Label: scab
  Total samples: 965
  Correctly classified: 611 (63.32%)
  Labeled as OOD: 79 (8.19%)
Label: rust
  Total samples: 372
  Correctly classified: 265 (71.24%)
  Labeled as OOD: 18 (4.84%)
Label: frog_eye_leaf_spot
  Total samples: 636
  Correctly classified: 632 (99.37%)
  Labeled as OOD: 2 (0.31%)
Label: powdery_mildew
  Total samples: 237
  Correctly classified: 216 (91.14%)
  Labeled as OOD: 2 (0.84%)
